# Overnight training: bigger / higher-resolution members, and training on test-like frames

Two new ensemble members, trained unattended with `run_queue.py` (one after the other, never at the same time):

| Run | Model | Input | Data | Epochs | Question |
|---|---|---|---|---|---|
| `yolo26m_1280_testres` | YOLO26**m** (COCO weights; Person B's runs are all YOLO26**s**) | 1280 px, scale jitter 0.25 | original training frames **+ the same frames shrunk to test resolutions** | 12 | Does a larger YOLO that has also seen test-like, low-resolution frames find more tiny objects? |
| `rtdetr_l_1280_testres` | RT-DETR-L, 500 queries (Person B's best recipe, `final_960_s025`) | **1280 px** (B: 960) | original + test-like | 12 | Does the strongest family gain from higher resolution (as YOLO did, 640 → 960 → 1280) and from test-like frames? |

**Why test-like frames.** Test clips are 680–1920 px wide (median ~1360), the training clips 1344–2720 px. Every model rescales its input to a fixed size, so a 680 px test clip reaches the model upscaled and blurry, with objects of a few pixels — a regime the training frames never show. The extra copy shrinks each training clip as a whole (like test, one resolution per clip) to a long side drawn from the test clips' widths, always smaller than its own. Test frames are also visibly blurry and compressed, so each shrunk clip also gets a mild Gaussian blur (σ 0.3–1.2 px) and JPEG compression (quality 60–90), drawn once per clip. YOLO labels are fractions of the image size, so they stay valid; boxes under 2 px after shrinking are dropped (same rule as `01_preprocess.ipynb`).

**Schedules.** B's 4-epoch YOLO runs were still improving and the 30-epoch `final_1280` overfit after ~epoch 16, so YOLO gets 12 epochs. B's RT-DETR `final_960_s025` peaked at epoch 31 of 40; here an epoch is 2× the frames (original + test-like) at ~1.8× the pixels, so 12 epochs (~24 passes over the original frames) with the same schedule shape (warm-up, mosaic off for the last epochs). Both are scored with the shared evaluator on the original val frames, like every other run.

**Run unattended** (from `src/`, nothing else on the GPU; estimated 3–4 h + 4–5 h on the RX 6800 XT):
```bash
WORKERS=2 nohup ../../.venv/bin/python run_queue.py experiments/train_overnight.ipynb:yolo26m_1280_testres,rtdetr_l_1280_testres > ../runs/overnight.log 2>&1 &
```

## Setup

In [ ]:
import json
import os
import sys
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml

# Works both when opened in VS Code (cwd = src/experiments) and from run_queue.py (cwd = src)
SRC_DIR = Path.cwd() if (Path.cwd() / "common").exists() else Path.cwd().parent
ROOT = SRC_DIR.parent
sys.path.insert(0, str(SRC_DIR))
import ultralytics
from ultralytics import RTDETR, YOLO

from common.data import CLASSES
from common.yolo import train_and_score

DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
PROC_ROOT = ROOT / "data_processed"
GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
FIXTURE = os.environ.get("FIXTURE") == "1"   # tiny CPU smoke test; never report its numbers
RUNS_DIR = ROOT / "runs" / ("overnight_fixture" if FIXTURE else "overnight")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "| fixture:", FIXTURE)

## Test-like copy of the training frames

Built once into `data_processed/train_testres/` (images + YOLO labels, same folder layout as `train/`); later runs reuse it.

In [ ]:
MIN_BOX_PX = 2


def build_testres(proc_root, test_csv, seed=42):
    out = proc_root / "train_testres"
    done = out / "stats.json"
    if done.exists():
        return pd.read_json(done)
    test = pd.read_csv(test_csv)
    test_sides = test.groupby("clip")[["width", "height"]].first().max(axis=1).to_numpy()   # one value per test clip
    images = json.loads((proc_root / "train" / "annotations_coco.json").read_text())["images"]
    by_clip = defaultdict(list)
    for im in images:
        by_clip[im["video"]].append(im)
    rng = np.random.default_rng(seed)
    rows = []
    for clip, ims in sorted(by_clip.items()):
        native = max(ims[0]["width"], ims[0]["height"])
        options = test_sides[test_sides < native]
        long_side = int(rng.choice(options)) if len(options) else native
        s = long_side / native
        sigma = float(rng.uniform(0.3, 1.2))          # mild blur, in pixels of the shrunk frame
        quality = int(rng.integers(60, 91))           # JPEG compression like a re-encoded video
        kept = dropped = 0
        for im in ims:
            rel = Path(im["file_name"])                                   # images/<clip>/<frame>.jpg
            img = cv2.imread(str(proc_root / "train" / rel))
            h, w = img.shape[:2]
            nw, nh = round(w * s), round(h * s)
            dst = out / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            small = cv2.GaussianBlur(cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA), (0, 0), sigma)
            cv2.imwrite(str(dst), small, [cv2.IMWRITE_JPEG_QUALITY, quality])
            src_lab = proc_root / "train" / "labels" / rel.relative_to("images").with_suffix(".txt")
            dst_lab = out / "labels" / rel.relative_to("images").with_suffix(".txt")
            dst_lab.parent.mkdir(parents=True, exist_ok=True)
            lines = src_lab.read_text().splitlines() if src_lab.exists() else []
            keep = [ln for ln in lines if ln.strip() and float(ln.split()[3]) * nw >= MIN_BOX_PX and float(ln.split()[4]) * nh >= MIN_BOX_PX]
            dst_lab.write_text("\n".join(keep) + ("\n" if keep else ""))
            kept += len(keep); dropped += len(lines) - len(keep)
        rows.append({"clip": clip, "native_long": native, "testres_long": long_side, "blur_sigma": round(sigma, 2),
                     "jpeg_quality": quality, "frames": len(ims),
                     "boxes_kept": kept, "boxes_dropped_under_2px": dropped})
    stats = pd.DataFrame(rows)
    stats.to_json(done)
    return stats


testres_stats = build_testres(PROC_ROOT, TEST_CSV)
print(f"{testres_stats.frames.sum()} frames shrunk; boxes kept {testres_stats.boxes_kept.sum()}, "
      f"dropped (under {MIN_BOX_PX} px) {testres_stats.boxes_dropped_under_2px.sum()}")
print("long side of the test-like copy (frames):", testres_stats.groupby("testres_long")["frames"].sum().to_dict())


def data_yaml(train_dirs, name):
    cfg = {"path": str(PROC_ROOT.resolve()), "train": train_dirs, "val": "val/images", "nc": len(CLASSES), "names": list(CLASSES)}
    p = RUNS_DIR / f"data_{name}.yaml"
    p.write_text(yaml.safe_dump(cfg, sort_keys=False))
    return p


DATA = {"original": data_yaml(["train/images"], "original"),
        "original+testres": data_yaml(["train/images", "train_testres/images"], "testres")}
testres_stats

## Protocols and experiments

Copied from `07_yolo.ipynb` (YOLO) and `08_rtdetr.ipynb` (RT-DETR) so the runs stay comparable with Person B's, with only the changes in the table above.

In [ ]:
WORKERS = int(os.environ.get("WORKERS", 2))
PREDICT = dict(conf=0.001, iou=0.7, max_det=500, device=DEVICE, batch=4)
PROTOCOLS = {
    "yolo": dict(epochs=12, batch=0.70, score_every=1, val_max_images=None, predict=PREDICT, train=dict(
        optimizer="auto", seed=42, deterministic=True, amp=True, workers=WORKERS, cache=False,
        fliplr=0.5, flipud=0.0, degrees=0.0, copy_paste=0.0, nbs=16, plots=False, val=True, device=DEVICE,
        warmup_epochs=1, close_mosaic=3, save_period=2)),
    "rtdetr": dict(epochs=12, batch=2, score_every=1, val_max_images=None, predict=PREDICT, train=dict(
        optimizer="AdamW", lr0=1e-4, lrf=0.1, weight_decay=1e-4, seed=42, deterministic=False, amp=True,
        workers=WORKERS, cache=False, fliplr=0.5, flipud=0.0, degrees=0.0, copy_paste=0.0, nbs=16,
        plots=False, val=True, device=DEVICE, warmup_epochs=1, close_mosaic=3, save_period=2)),
}

# RT-DETR-L with 500 queries, as in 08_rtdetr.ipynb (the query count changes no weight shape, so COCO weights transfer)
_cfg = yaml.safe_load((Path(ultralytics.__file__).parent / "cfg/models/rt-detr/rtdetr-l.yaml").read_text())
_cfg["head"][-1][-1] = ["nc", 256, 500]
RTDETR_L_500Q = RUNS_DIR / "rtdetr-l-500q.yaml"
RTDETR_L_500Q.write_text(yaml.safe_dump(_cfg, sort_keys=False))

EXPERIMENTS = {   # family, data, and the train_and_score config
    "yolo26m_1280_testres": dict(family="yolo", data="original+testres",
                                 cfg=dict(model="yolo26m.pt", weights=None, imgsz=1280, train=dict(scale=0.25), predict={})),
    "rtdetr_l_1280_testres": dict(family="rtdetr", data="original+testres",
                          cfg=dict(model=str(RTDETR_L_500Q), weights=None, imgsz=1280,
                                   train=dict(pretrained="rtdetr-l.pt", scale=0.25), predict={})),
}
MODEL_CLS = {"yolo": YOLO, "rtdetr": RTDETR}
if FIXTURE:
    for p in PROTOCOLS.values():
        p.update(epochs=1, batch=2, val_max_images=4)
        p["train"].update(workers=0, amp=False, close_mosaic=0, warmup_epochs=0, save_period=1)
        p["predict"] = {**p["predict"], "batch": 2}
    for e in EXPERIMENTS.values():
        e["cfg"]["imgsz"] = 320
    EXPERIMENTS["yolo26m_1280_testres"]["cfg"]["model"] = "yolo26n.pt"

## Train

In [ ]:
RUN = ["yolo26m_1280_testres", "rtdetr_l_1280_testres"]   # unattended: run_queue.py experiments/train_overnight.ipynb:<names>
for name in RUN:
    e = EXPERIMENTS[name]
    train_and_score(name, {**e["cfg"], "data": e["data"]}, PROTOCOLS[e["family"]], DATA[e["data"]], RUNS_DIR, GT_FILE,
                    DATA_ROOT, MODEL_CLS[e["family"]])

## Results

Best epoch of each run (shared evaluator, original val frames), next to Person B's runs they extend.

In [ ]:
rows = {}
for d in [RUNS_DIR / n for n in EXPERIMENTS] + [ROOT / "runs/yolo/baseline", ROOT / "runs/yolo/final_1280", ROOT / "runs/rtdetr/final_960_s025"]:
    if (d / "summary.json").exists():
        rows[d.name] = json.loads((d / "summary.json").read_text())["best"]
pd.DataFrame(rows).T[["epoch", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]].round(4)

In [ ]:
# Per-epoch curves of the new runs
for n in EXPERIMENTS:
    h = RUNS_DIR / n / "history.json"
    if h.exists():
        print(n)
        display(pd.DataFrame(json.loads(h.read_text()))[["epoch", "mAP", "AP50", "AP_small"]].round(4).set_index("epoch").T)

**Observations:** _fill in_